In [1]:
### 0 ### Inizializzazione - eseguire solo una volta per "vita" del kernel
import sys
import logging
from pathlib import Path

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline_handler import PipelineHandler

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# Percorsi
sidecar_path = project_root / "data/processed/armstrong/full_test1.json"
queries_path = project_root / "data/queries/armstrong/eval_queries.json"

# Inizializzazione gestore pipeline
handler = PipelineHandler(
    sidecar_path=sidecar_path,
    queries_path=queries_path
)

print("!>> PipelineHandler e SidecarManager inizializzati con successo.")

/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(
07:06:24 [INFO] httpx: HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
07:06:25 [INFO] src.pipeline_handler: ?> Pipeline Inizializzata


!>> PipelineHandler e SidecarManager inizializzati con successo.


In [ ]:
### PRE-TAGGING ###
candidate_tags = ["Jazz", "Ciclismo", "Astronautica"]
batch_size = 100

handler.pretag_sidecar(
        sidecar_path=sidecar_path,
        candidate_tags=candidate_tags,
        batch_size=batch_size
    )

In [ ]:
### CONFIG. PRE-DET. QUERIES ###
pred_text = "Chi è Armstrong e quali sono i principali traguardi raggiunti nella sua carriera?"
query_id = "Q_AMBIG_01"

handler.add_predetermined_query(
        query_text=pred_text,
        query_id=query_id
    )

In [2]:
### 1 ### Impostazione Parametri
handler.set_parameters(
    top_j=7,
    top_k=10,
    top_n=7,
    order_by=["DIST", "TAGS"]
)

07:06:25 [INFO] src.pipeline_handler: ?> Parametri aggiornati: top_j=7, top_k=10, top_n=7, order_by=['DIST', 'TAGS']


In [3]:
### 2 ### Selezione Query

# Q_AMBIG_01, Q_AMBIG_02, Q_AMBIG_03, Q_SPEC_JAZZ_01, Q_SPEC_BIKE_01, Q_SPEC_ASTRO_01
#handler.update_query(new_id="Q_SPEC_ASTRO_01")

handler.update_query(new_text="Quali sono i principali traguardi raggiunti nella carriera di Armstrong?")

07:06:25 [INFO] src.pipeline_handler: ?> Testo Query aggiornato con il testo fornito 
07:06:25 [INFO] src.pipeline_handler: ?> Vettore embedding Query aggiornato: text:Quali sono i principali traguardi raggiunti nella carriera di Armstrong?
07:06:25 [INFO] src.pipeline_handler: ?> Verifica match della nuova query con le query pre-determinate
07:06:25 [INFO] src.predetermined_query_handling_methods: <<| Match vettoriale trovato per Query Pre-Determinata [predq_Q_AMBIG_01] con similarità 0.9375 (>= 0.85) |>>
07:06:25 [INFO] src.pipeline_handler:    >>> Query: <| Quali sono i principali traguardi raggiunti nella carriera di Armstrong? |> ricondotta a <| Chi è Armstrong e quali sono i principali traguardi raggiunti nella sua carriera? |> 


In [4]:
### 3 ### Tag query
conv_tag_list = ["Astronautica"]
ass_tag_list = ["Neil"]
auto_ass_flag = False

handler.update_query_tags(
    new_conv_list=conv_tag_list,
    new_ass_list=ass_tag_list,
    auto_ass_flag=auto_ass_flag,
    print_debug=True
)

07:06:25 [INFO] src.pipeline_handler: ?> Query vettorizzata. Tag assegnati: ['Astronautica', 'Neil']


!>> Tag assegnati alla query:
   >>> Tag di Conversazione: ['Astronautica']
   >>> Tag Manuali della Query: ['Neil']
   >>> Assegnazione Tag Automatici disabilitata
<<< Lista completa: ['Astronautica', 'Neil']


In [5]:
### 4 ### Retrieval Vettoriale
handler.run_vectorial_retrieval()

07:06:25 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/multilanguage_armstrong/points/query "HTTP/1.1 200 OK"
07:06:25 [INFO] src.pipeline_handler: ?> Recuperati 10 record con relativi vettori di embedding


In [6]:
### 5 ### Reranking
handler.run_top_j_distance_tag_reranking(print_debug=True)

07:06:25 [INFO] src.predetermined_query_handling_methods: <<| Match vettoriale trovato per Query Pre-Determinata [predq_Q_AMBIG_01] con similarità 0.9375 (>= 0.85) |>>
07:06:25 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/multilanguage_armstrong/points "HTTP/1.1 200 OK"


?> [DEBUG INFO] all_reranked:
         >> Il Reranker ha restituito 13 record
    >>> cid: ['neil_armstrong_1']
        |> category: [1]
           |> initial_score: [0.5815]
           |> final_score: [0.8723]
    >>> cid: ['neil_armstrong_3']
        |> category: [1]
           |> initial_score: [0.5679]
           |> final_score: [0.6815]
    >>> cid: ['neil_armstrong_7']
        |> category: [1]
           |> initial_score: [0.5852]
           |> final_score: [0.5852]
    >>> cid: ['neil_armstrong_2']
        |> category: [1]
           |> initial_score: [0.5594]
           |> final_score: [0.5034]
    >>> cid: ['neil_armstrong_5']
        |> category: [2]
           |> initial_score: [0.5569]
           |> final_score: [1.0442]
    >>> cid: ['neil_armstrong_4']
        |> category: [2]
           |> initial_score: [0.498]
           |> final_score: [0.7469]
    >>> cid: ['neil_armstrong_6']
        |> category: [2]
           |> initial_score: [0.4773]
           |> final_score: [

In [7]:
import ipywidgets as widgets
from IPython.display import display

print("I Grafi che contano veramente nella pipeline finale (dunque escludendo quelli presenti per DEBUG) sono:")
print("  > 1° Retrieval (Vettoriale)")
print("  > Top-J Reranked")

# Mappatura tra i nomi visibili nei pulsanti e le chiavi interne delle fasi
PHASE_MAP = {
    "1° Retrieval (Vettoriale)": "vectorial",
    "2° Retrieval (Distanze)": "distance",
    "Top-J Reranked": "reranked",
    "All Reranked": "all_reranked"
}

# Selettore orizzontale delle fasi (ToggleButtons)
phase_selector = widgets.ToggleButtons(
    options=list(PHASE_MAP.keys()),
    description='Fase Grafo:',
    disabled=False,
    button_style='info',
    tooltips=[
        'Mostra i Top-K nodi del retrieval vettoriale iniziale',
        'Mostra i nodi iniziali uniti ai vicini recuperati per distanza',
        'Mostra i Top-J nodi selezionati dal Reranker per l\'LLM',
        'Mostra l\'intero elenco ordinato dal Reranker'
    ],
)

# Container per il rendering dinamico del widget
output_area = widgets.Output()

def on_phase_change(change):
    selected_label = change['new']
    phase_key = PHASE_MAP[selected_label]
    
    with output_area:
        output_area.clear_output(wait=True)
        handler.update_widget_view(phase_key)
        display(handler.widget)

# Registrazione dell'evento sul cambio di selezione
phase_selector.observe(on_phase_change, names='value')

# Inizializzazione della vista sulla prima fase
handler.update_widget_view("vectorial")

# Visualizzazione dell'interfaccia a schermo
display(phase_selector)
display(output_area)

# Popola l'output area con la vista iniziale del grafo
with output_area:
    display(handler.widget)

07:06:26 [INFO] src.pipeline_handler: ?> Widget aggiornato alla fase 'vectorial' (11 nodi)


I Grafi che contano veramente nella pipeline finale (dunque escludendo quelli presenti per DEBUG) sono:
  > 1° Retrieval (Vettoriale)
  > Top-J Reranked


ToggleButtons(button_style='info', description='Fase Grafo:', options=('1° Retrieval (Vettoriale)', '2° Retrie…

Output()

In [8]:
### LLM ### Generazione risposte LLM
comparison = handler.compare_retrieval_vs_reranking()

baseline = comparison["vectorial_baseline"]
reranked = comparison["post_reranking"]

print("=" * 80)
print(f"???> Query: '{baseline['query']}'")
print("=" * 80)

print("\n --- 1. RISPOSTA BASELINE (1° Retrieval Vettoriale / Raw Records) ---")
print(f"  >> Chunk inclusi ({len(baseline['chunk_ids'])}): {baseline['chunk_ids']}")
print("\n  >> PROMPT INVIATO:")
print("-" * 40)
print(baseline["prompt"])
print("-" * 40)
print("\n  << RISPOSTA LLM:")
print(baseline["response"])

print("\n" + "=" * 80)

print("\n --- 2. RISPOSTA RERANKED (Post DistanceTagReranking / Top-J) ---")
print(f"  >> Chunk inclusi ({len(reranked['chunk_ids'])}): {reranked['chunk_ids']}")
print("\n  >> PROMPT INVIATO:")
print("-" * 40)
print(reranked["prompt"])
print("-" * 40)
print("\n  << RISPOSTA LLM:")
print(reranked["response"])
print("=" * 80)

07:06:26 [INFO] src.pipeline_handler: ?> Invocazione LLM
07:06:38 [INFO] httpx: HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
07:06:49 [INFO] src.pipeline_handler: ?> Invocazione LLM
07:06:49 [INFO] httpx: HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


???> Query: 'Quali sono i principali traguardi raggiunti nella carriera di Armstrong?'

 --- 1. RISPOSTA BASELINE (1° Retrieval Vettoriale / Raw Records) ---
  >> Chunk inclusi (10): ['lance_armstrong_2', 'louis_armstrong_6', 'lance_armstrong_3', 'lance_armstrong_1', 'neil_armstrong_7', 'neil_armstrong_1', 'lance_armstrong_7', 'lance_armstrong_5', 'neil_armstrong_3', 'neil_armstrong_2']

  >> PROMPT INVIATO:
----------------------------------------
Sei un assistente di ricerca. Rispondi alla domanda seguente basandoti ESCLUSIVAMENTE sul contesto fornito. Se il contesto non contiene informazioni sufficienti o è parziale, evidenzialo chiaramente senza inventare fatti non presenti nelle fonti.

Domanda: Quali sono i principali traguardi raggiunti nella carriera di Armstrong?

Contesto recuperato (10 chunk):
[CHUNK 1 | ID: lance_armstrong_2 | Score: 0.6533]
Nel mondo del ciclismo professionistico, Lance Armstrong è ricordato sia per i suoi successi nelle corse a tappe sulle Alpi e sui Pire